In [ ]:
# ====================== GPU PICKER (server-friendly) ======================
# Set which GPU to use (0, 1, ...). Set to None to NOT force anything (respects the environment).
GPU_ID = 1 # Change to None when you want to leave it free for other users
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
torch.backends.cudnn.benchmark = True  # if the size varies A LOT, consider False
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.set_device(0)
print("Device:", device, "| Visible devices (after mapping):", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if device == "cuda":
    try:
        print("Using mapped GPU:", torch.cuda.get_device_name(0))
    except Exception:
        pass
    for i in range(torch.cuda.device_count()):
        try:
            print(f"Device {i}:", torch.cuda.get_device_name(i))
        except Exception:
            pass
import sys
from datetime import datetime
import time

In [ ]:
import sys
sys.path.insert(0, "/workspace/app")
import matplotlib.pyplot as plt
import pandas as pd
import random
import shap
import numpy as np
import os
import ast
import pickle
import importlib
from coding.Data_Processing import Run_RNN as rf
importlib.reload(rf)
from coding.Data_Preprocessing import DataPreprocessing_melt as mf
from coding.Data_Processing import TargetVariables as tv
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
from torch.cuda.amp import autocast
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90  

___

LOG

In [ ]:
class Tee(object):
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

In [ ]:
log_path = f"COLA_BIN_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
orig_stdout = sys.stdout
orig_stderr = sys.stderr
log_file = open(log_path, "w", buffering=1) 
sys.stdout = Tee(orig_stdout, log_file)
sys.stderr = Tee(orig_stderr, log_file)
start_time = time.perf_counter()
print("---- Logging started ----")
print("Log file:", log_path)

___

Dataset

In [ ]:
df_wav = pd.read_pickle('/workspace/app/planilhas/df_emb3D.pkl') # wav2vec
df_wav['Patient_ID'] = encoder.fit_transform(df_wav['Patient_ID'])
print(f"Patients: {df_wav['Patient_ID'].nunique()}")

In [ ]:
df_wav = tv.standardized_binary_evolution(df_wav, MAX_HDRS, MAX_CDI, 40, 60)
df_wav['Y_Binary_Classe_Delta_Y'] = df_wav['Y_Binary_Classe_Delta_Y'].map({'Worse': 0, 'Better': 1})
df_wav = df_wav.dropna(subset=['Y_Binary_Classe_Delta_Y'])
df_wav['Y_Binary_Classe_Delta_Y'] = df_wav['Y_Binary_Classe_Delta_Y'].astype(int)
print(df_wav["Y_Binary_Classe_Delta_Y"].value_counts())

___

Preprocessing

In [ ]:
suffixes = ('Date', 'F1_Score', 'F2_Score', 'F3_Score', '_Days', 'soc_Score', 'iso_Score', 'sup_Score')
prefixes = ('Questionnary1', 'Questionnary2', 'LogMel_Pat_Speech', 'Days_')
exact_cols = {
    "Age", 'Patient_Gender', 'Randomization_Group', 'Education', 'Number_Complete_Sessions',
    'Questionnary5_HQ_25_T0_TOT_Score', 'Questionnary5_HQ_25_T1_TOT_Score',
    'Questionnary3_TAS_20_T1_TOT_Score', 'Questionnary4_AQC_T1_TOT_Score'} # Alexithymia T1 to avoid leakage
cols_to_drop = [
    col for col in df_wav.columns 
    if col in exact_cols or col.endswith(suffixes) or col.startswith(prefixes)]
df_wav.drop(columns=cols_to_drop, inplace=True)

In [ ]:
def get_embeddings_per_segment_mean_3D(df, meta_cols):
    embedding_cols = [col for col in df.columns if col.startswith("Embeddings_Session")]

    if not embedding_cols:
        raise ValueError("No Embeddings_Session columns were found.")

    df_long = df.melt(
        id_vars=meta_cols, value_vars=embedding_cols,
        var_name="Embedding_feature", value_name="Embedding_npy"
    )

    extracted = df_long["Embedding_feature"].str.extract(
        r"Embeddings_Session(\d+)_Segment(\d+)"
    )
    df_long["Session"] = pd.to_numeric(extracted[0], errors="coerce")
    df_long["Segment"] = pd.to_numeric(extracted[1], errors="coerce")

    df_long = df_long.dropna(
        subset=["Session", "Segment", "Embedding_npy"]
    ).copy()

    df_long["Session"] = df_long["Session"].astype(int)
    df_long["Segment"] = df_long["Segment"].astype(int)

    new_order = ["Patient_ID", "Session", "Segment"]
    new_order += [col for col in meta_cols if col != "Patient_ID"]
    new_order += ["Embedding_npy"]

    df_long = df_long[new_order].reset_index(drop=True)

    duplicated = df_long.duplicated(
        subset=["Patient_ID", "Session", "Segment"]
    )
    if duplicated.any():
        raise ValueError(
            f"{duplicated.sum()} duplicated patient/session/segment rows found."
        )

    return df_long

wav2vec

In [ ]:
meta = ['Patient_ID', 'Questionnary3_TAS_20_T0_TOT_Score', 'Questionnary4_AQC_T0_TOT_Score', 'Y_Binary_Classe_Delta_Y']
#df_wav = mf.get_embeddings_per_segment_mean_3D(df_wav, meta)
df_wav = get_embeddings_per_segment_mean_3D(df_wav, meta)

log-mel

In [ ]:
df_cnn = pd.read_pickle("/workspace/app/planilhas/df_cnn_segments_bin.pkl")

___

### Experiments

In [ ]:
unique_patients = df_cnn['Patient_ID'].unique()
target = 'Y_Binary_Classe_Delta_Y'
alex_col="Alexithymia_T0"
num_epochs = 20
BS = 2
lambda_alex = 0.05
tau = 0.5
pw = None
#beta_kl = 0.0
#stoc = False

In [ ]:
def save_binary_results(results, model_name, out_root="/workspace/app/planilhas/TaCoLa"):
    df_results = pd.DataFrame(results)
    result_path = os.path.join(out_root, f"BIN_{model_name}.xlsx")
    summary_path = os.path.join(out_root, f"BIN_{model_name}_SUMMARY.xlsx")
    df_results.to_excel(result_path, index=False)
    cols = ["y_patient_true", "y_patient_pred", "p_patient"]
    for col in cols:
        df_results[col] = pd.to_numeric(df_results[col], errors="coerce")

    valid = np.isfinite(df_results[cols]).all(axis=1)
    invalid = df_results.loc[~valid].copy()
    d = df_results.loc[valid].copy()

    print(f"Total folds: {len(df_results)} | Valid: {len(d)} | Invalid: {len(invalid)}")
    if not invalid.empty:
        show_cols = [c for c in ["Patient_ID", "Test_Loss", "Best_Epoch", *cols]
                     if c in invalid.columns]
        print(invalid[show_cols])

    if d.empty:
        raise ValueError(f"No valid folds found for {model_name}.")
    y_true = d["y_patient_true"].astype(int).values
    y_pred = d["y_patient_pred"].astype(int).values
    y_prob = d["p_patient"].astype(float).values
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    auroc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan
    df_summary = pd.DataFrame([{
        "Model": model_name,
        "N_Total": len(df_results),
        "N_Valid": len(d),
        "N_Invalid": len(invalid),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_Pos": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "Precision_Pos": precision_score(y_true, y_pred, zero_division=0),
        "F1_Pos": f1_score(y_true, y_pred, zero_division=0),
        "AUROC": auroc
    }])
    print(df_summary)
    #df_summary.to_excel(summary_path, index=False)
    return df_results, df_summary

#### Experiment 2 
a) Alexithymia-Query Conditioned Longitudinal Attention (AQ-CoLA) - Binary usng CNN-log
- mode="full"
- trait_source="alex" | Standardized alexithymia within the fold enters the network, is transformed into a z-trait, and participates in the attention query

The alexithymia used in this variant should be the Alexithymia_T0 constructed with mean and standard deviation calculated only in the training patients of each fold.

* obs.: For trait_source="speech":  The query comes from the acoustic representation of the first session. Alexithymia is used only as an auxiliary supervision tool. - LOSS

In [ ]:
name = "COLA_Alex_LogMel_Full"
print(f"Processing Dataset {name} Binary")
results_bin = [
    rf.class_lopoRNN(
        p, df_cnn, "Patient_ID", target, "CNN_SegEmb_npy", model_name="COLA",
        modeCola="full", representation="cnn", epochs=num_epochs, BS=BS,
        alex_col=alex_col, device=device, pos_weight=pw, trait_source="alex",
        lambda_alex=lambda_alex, tau=tau
    )
    for p in unique_patients
]
print(f"{name} Done")
df_results, df_summary = save_binary_results(results_bin, name)

b) Alexithymia-Query Conditioned Longitudinal Attention (AQ-CoLA) - Binary usng wav2vec

In [ ]:
name = "COLA_Alex_Wav_Full"
print(f"Processing Dataset {name} Binary")
results_bin = [
    rf.class_lopoRNN(p, df_wav, "Patient_ID", target, 'Embedding_npy', model_name = "COLA",
            modeCola = "full", representation="wav2vec", epochs=num_epochs, BS=BS, alex_col=alex_col,
            device=device, pos_weight=pw, trait_source="alex", lambda_alex=lambda_alex, tau=tau
    )
    for p in unique_patients
]
print(f"{name} Done")
df_results, df_summary = save_binary_results(results_bin, name)

___

#### Experiment 3 - CoLA Ablations (Binary)
Representation: Use the best from Experiment 2 (CNN-log or wav2vec)

1) Ablations of the CoLA approach

trait_source = "alex"

Compare:
* trait:       [h, z]
* interaction: [h, z, h*z]
* delta_h1:    [h, z, h-h1]
* full:        [h, z, h*z, h-h1]

In [ ]:
dic_path = {
    "path_cnn": "/workspace/app/planilhas/TaCoLa/BIN_COLA_Alex_LogMel_Full_SUMMARY.xlsx",
    "path_wav": "/workspace/app/planilhas/TaCoLa/BIN_COLA_Alex_Wav_Full_SUMMARY.xlsx"
}
if os.path.isfile(dic_path["path_cnn"]) and os.path.isfile(dic_path["path_wav"]):
    df_cnn_SUM = pd.read_excel(dic_path["path_cnn"])
    df_wav_SUM = pd.read_excel(dic_path["path_wav"])

    f1_cnn = pd.to_numeric(df_cnn_SUM["F1_Pos"], errors="coerce").iloc[0]
    f1_wav = pd.to_numeric(df_wav_SUM["F1_Pos"], errors="coerce").iloc[0]

    if f1_cnn > f1_wav:
        best_representation = "cnn"
        best_df = df_cnn.copy()
    else:
        best_representation = "wav2vec"
        best_df = df_wav.copy()

    print("CNN F1:", f1_cnn)
    print("wav2vec F1:", f1_wav)
    print("Best representation:", best_representation)

In [ ]:
best_representation = "cnn"
representation = best_representation
best_df = df_cnn.copy()
npy_col = "CNN_SegEmb_npy" if best_representation == "cnn" else "Embedding_npy"
name_best = "COLA_Alex_LogMel" if best_representation == "cnn" else "COLA_Alex_Wav"

In [ ]:
experiments = [
    {"name": f"{name_best}_Trait", "modeCola": "trait"},
    {"name": f"{name_best}_Inter", "modeCola": "interaction"},
    {"name": f"{name_best}_DELTAH1", "modeCola": "delta_h1"}
]
for exp in experiments:
    print(f"Processing Dataset {exp['name']} Binary")
    results_bin = [
        rf.class_lopoRNN(
            p, best_df, "Patient_ID", target, npy_col, model_name="COLA",
            alex_col=alex_col, modeCola=exp["modeCola"],
            representation=best_representation, epochs=num_epochs, BS=BS,
            device=device, pos_weight=pw, trait_source="alex",
            lambda_alex=lambda_alex, tau=tau
        )
        for p in unique_patients
    ]
    print(f"{exp['name']} Done")
    df_results, df_summary = save_binary_results(results_bin, exp['name'])

Seeds

In [ ]:
for seed in [42, 123, 2026]:
    print(f"Processing COLA_Alex_LogMel_Trait | seed={seed}")
    results_bin = [
        rf.class_lopoRNN(p, df_cnn, "Patient_ID", target, "CNN_SegEmb_npy", model_name="COLA", modeCola="trait", representation="cnn",
            epochs=num_epochs, BS=BS, alex_col=alex_col, device=device, trait_source="alex", lambda_alex=lambda_alex, tau=0.5, seed=seed)
        for p in unique_patients
    ]
    pd.DataFrame(results_bin).to_excel(f"/workspace/app/planilhas/TaCoLa/Seeds/BIN_COLA_Alex_LogMel_Trait_seed{seed}.xlsx", index=False)
    print(f"COLA_Alex_LogMel_Trait | seed={seed} Done")

2) Trait Origin
Fix the best mode and compare:

trait_source="alex"

trait_source="speech"

* This answers:
Is measured baseline alexithymia more useful than a speech-derived latent trait supervised by alexithymia?

This comparison separates two truly different hypotheses.

In [ ]:
best_modeCola = "trait" # To be filled in with the best combination from Exp 2 and Exp 3.1
print(f"Processing Dataset COLA_Speech Binary")
results_bin = [rf.class_lopoRNN(p, best_df, "Patient_ID", target, npy_col, model_name = "COLA", alex_col=alex_col,
                         modeCola = best_modeCola, representation=representation, epochs=num_epochs, BS=BS,
                         device=device, pos_weight=pw, trait_source="speech", lambda_alex=lambda_alex, tau=tau)
        for p in unique_patients]
print(f"COLA_Speech Done")
df_results, df_summary = save_binary_results(results_bin, "COLA_Speech")

Seed for Speech

In [ ]:
best_modeCola = "trait" 
for seed in [42, 123, 2026]:
    print(f"Processing Dataset COLA_Speech Binary | seed={seed}")
    results_bin = [rf.class_lopoRNN(p, df_cnn, "Patient_ID", target, "CNN_SegEmb_npy", model_name = "COLA", alex_col=alex_col,
                            modeCola = "trait", representation="cnn", epochs=num_epochs, BS=BS,
                            device=device, pos_weight=pw, trait_source="speech", lambda_alex=lambda_alex, tau=0.5, seed=seed)
            for p in unique_patients]
    pd.DataFrame(results_bin).to_excel(f"/workspace/app/planilhas/TaCoLa/Seeds/BIN_COLA_Speech_seed{seed}.xlsx", index=False)
    print(f"COLA_Alex_LogMel_Trait | seed={seed} Done")

___

LOG

In [ ]:
elapsed_time = time.perf_counter() - start_time
print("\n---- Experiment finished ----")
print(f"Total execution time: {elapsed_time:.2f} seconds")
print(f"Total execution time: {elapsed_time / 60:.2f} minutes")